# API Gateway → Lambda → Firehose 로그 수집
이 노트북은 API Gateway에서 들어오는 이벤트 로그를 Lambda에서 정규화한 뒤 Kinesis Firehose로 전달하는 코드 예제입니다.

In [ ]:
import jsonimport boto3import osimport base64import loggingfrom typing import Any, Dict, List# ===============================# 기본 설정# ===============================logger = logging.getLogger()logger.setLevel(logging.INFO)firehose = boto3.client("firehose")DELIVERY_STREAM_NAME = os.environ["DELIVERY_STREAM_NAME"]EXPECTED_API_KEY = os.environ["EXPECTED_API_KEY"]# 🔥 누락되어 있던 상수 (필수)MAX_EVENTS = 500# ===============================# 공통 응답 함수# ===============================def response(status_code: int, body: Any) -> Dict[str, Any]:    return {        "statusCode": status_code,        "headers": {            "Content-Type": "application/json",            "Access-Control-Allow-Origin": "*"        },        "body": json.dumps(body)    }# ===============================# 이벤트 정규화# ===============================def normalize_events(body: Any) -> List[Dict[str, Any]]:    if not isinstance(body, dict) or len(body) != 1:        raise ValueError("invalid root schema")    user_id, activities = next(iter(body.items()))    if not isinstance(activities, dict):        raise ValueError("invalid user payload")    normalized_events: List[Dict[str, Any]] = []    for activity, events in activities.items():        if not isinstance(events, list):            raise ValueError(f"invalid activity payload: {activity}")        for evt in events:            if not isinstance(evt, dict):                raise ValueError("event must be an object")            normalized_events.append({                "_user_id": user_id,                "_activity": activity,                **evt            })    if not normalized_events:        raise ValueError("no events found")    return normalized_events# ===============================# Lambda Handler# ===============================def lambda_handler(event, context):    request_id = context.aws_request_id    logger.info({        "message": "request received",        "request_id": request_id    })    # 1️⃣ 인증    headers = event.get("headers") or {}    api_key = (        headers.get("x-api-key")        or headers.get("X-Api-Key")        or headers.get("X-API-KEY")    )    if api_key != EXPECTED_API_KEY:        logger.warning({"message": "unauthorized", "request_id": request_id})        return response(403, {"error": "Forbidden"})    # 2️⃣ Body 디코딩    raw_body = event.get("body", "")    if event.get("isBase64Encoded"):        raw_body = base64.b64decode(raw_body).decode("utf-8")    # 3️⃣ JSON 파싱    try:        body = json.loads(raw_body)    except json.JSONDecodeError:        return response(400, {"error": "Invalid JSON"})    # 4️⃣ 정규화    try:        events = normalize_events(body)    except ValueError as e:        return response(400, {"error": str(e)})    if len(events) > MAX_EVENTS:        return response(413, {"error": "Too many events"})    # 5️⃣ Firehose 전송    records = [{"Data": json.dumps(evt, ensure_ascii=False) + "\n"} for evt in events]    logger.info({        "message": "sending to firehose",        "request_id": request_id,        "count": len(records),        "stream": DELIVERY_STREAM_NAME    })    resp = firehose.put_record_batch(        DeliveryStreamName=DELIVERY_STREAM_NAME,        Records=records    )    failed = resp.get("FailedPutCount", 0)    if failed > 0:        logger.error({            "message": "firehose put failed",            "failed": failed,            "responses": resp.get("RequestResponses")        })        return response(500, {"error": "Firehose put failed", "failed": failed})    return response(200, {        "received": len(events),        "failed": 0    })